In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
import xgboost as xgb
import torch
import warnings

warnings.filterwarnings('ignore')

# Device selection
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 4090


In [2]:
# Load data
print("Loading data...")
train_df = pd.read_csv('dataset/train_v2.csv')
test_df = pd.read_csv('dataset/test_v2.csv')

print(f"Training set: {train_df.shape}")
print(f"Test set: {test_df.shape}")

# Feature and target columns
feature_cols = [f'feature_{i}' for i in range(1, 31)]
target_cols = ['target_short', 'target_medium', 'target_long']
TARGET_WEIGHTS = {'short': 0.5, 'medium': 0.3, 'long': 0.2}

print(f"Features: {len(feature_cols)}")
print(f"Targets: {target_cols}")
print(f"Weights: {TARGET_WEIGHTS}")

Loading data...
Training set: (139392, 37)
Test set: (34348, 34)
Features: 30
Targets: ['target_short', 'target_medium', 'target_long']
Weights: {'short': 0.5, 'medium': 0.3, 'long': 0.2}


In [3]:
train_df.replace([np.inf, -np.inf], np.nan, inplace=True)
test_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    
# 对每个 feature_i，计算当日 z-score
for i in range(1, 31):
    col = f'feature_{i}'
    # 注意：只用训练集统计量！避免泄露
    train_df[f'{col}_z'] = train_df.groupby('date_id')[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-5)
    )
    
    # 测试集：按 test 自身的 date_id 计算（比赛允许）
    test_df[f'{col}_z'] = test_df.groupby('date_id')[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-5)
    )
    
feature_cols += [f'feature_{i}_z' for i in range(1, 31)]

train_df['minute_sin'] = np.sin(2 * np.pi * train_df['minute_id'])
train_df['minute_cos'] = np.cos(2 * np.pi * train_df['minute_id'])
test_df['minute_sin'] = np.sin(2 * np.pi * test_df['minute_id'])
test_df['minute_cos'] = np.cos(2 * np.pi * test_df['minute_id'])

feature_cols += ['minute_sin', 'minute_cos']

for i in range(1, 31):
    col = f'feature_{i}'
    train_df[f'{col}_nan'] = train_df[col].isna().astype(int)
    test_df[f'{col}_nan'] = test_df[col].isna().astype(int)
    feature_cols.append(f'{col}_nan')
    
# train_df[f'{col}_z'] = train_df[f'{col}_z'].clip(-5, 5)
# test_df[f'{col}_z'] = test_df[f'{col}_z'].clip(-5, 5)

In [4]:
# 正相关特征的交互
train_df['feat8z_x_feat27z'] = train_df['feature_8_z'] * train_df['feature_27_z']
test_df['feat8z_x_feat27z'] = test_df['feature_8_z'] * test_df['feature_27_z']
feature_cols.append('feat8z_x_feat27z')

# 负相关特征的交互
train_df['feat10z_x_feat6z'] = train_df['feature_10_z'] * train_df['feature_6_z']
test_df['feat10z_x_feat6z'] = test_df['feature_10_z'] * test_df['feature_6_z']
feature_cols.append('feat10z_x_feat6z')

# 差值交互
train_df['feat8z_minus_feat27z'] = train_df['feature_8_z'] - train_df['feature_27_z']
test_df['feat8z_minus_feat27z'] = test_df['feature_8_z'] - test_df['feature_27_z']
train_df['feat10z_minus_feat6z'] = train_df['feature_10_z'] - train_df['feature_6_z']
test_df['feat10z_minus_feat6z'] = test_df['feature_10_z'] - test_df['feature_6_z']
feature_cols.extend(['feat8z_minus_feat27z', 'feat10z_minus_feat6z'])

# 比值交互（防止除零）
epsilon = 1e-5
train_df['feat8z_div_feat27z'] = train_df['feature_8_z'] / (train_df['feature_27_z'] + epsilon)
test_df['feat8z_div_feat27z'] = test_df['feature_8_z'] / (test_df['feature_27_z'] + epsilon)
train_df['feat10z_div_feat6z'] = train_df['feature_10_z'] / (train_df['feature_6_z'] + epsilon)
test_df['feat10z_div_feat6z'] = test_df['feature_10_z'] / (test_df['feature_6_z'] + epsilon)
feature_cols.extend(['feat8z_div_feat27z', 'feat10z_div_feat6z'])

In [5]:
split_idx = int(len(train_df) * 0.7)
gap = 0
gap = 20000

X_train = train_df[feature_cols].iloc[:split_idx].values
X_val = train_df[feature_cols].iloc[split_idx+gap:].values

y_train = train_df[target_cols].iloc[:split_idx].values
y_val = train_df[target_cols].iloc[split_idx+gap:].values

X_test = test_df[feature_cols].values

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
assert not np.isinf(train_df[feature_cols]).any().any(), "Train has inf!"
assert not np.isinf(test_df[feature_cols]).any().any(), "Test has inf!"

Train: (97574, 98), Val: (21818, 98), Test: (34348, 98)


In [7]:
print("="*80)
print("Training XGBoost Models")
print("="*80)

xgb_models = {}
xgb_val_predictions = {}

for i, target_name in enumerate(['short', 'medium', 'long']):
    print(f"\n--- Training XGBoost for target_{target_name} ---")
    
    xgb_params = {
        'objective': 'reg:squarederror',  # ← 更匹配 MAE 目标！
        'tree_method': 'hist',
        'device': 'cuda:1',
        'max_depth': 5,                    
        'learning_rate': 0.02,             
        'n_estimators': 1000,              
        'subsample': 0.9,
        'colsample_bytree': 0.7,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,                 
        'random_state': 42,
        'eval_metric': 'mae',
        'early_stopping_rounds': 50
    }
    
    model = xgb.XGBRegressor(**xgb_params)
    
    # Train
    model.fit(
        X_train, y_train[:, i],
        eval_set=[(X_val, y_val[:, i])],
        verbose=100
    )
    
    # Predict on validation set
    val_pred = model.predict(X_val)
    val_mae = mean_absolute_error(y_val[:, i], val_pred)
    
    print(f"Best iteration: {model.best_iteration}")
    print(f"Validation MAE: {val_mae:.6f}")
    
    # Store model and predictions
    xgb_models[target_name] = model
    xgb_val_predictions[target_name] = val_pred

# Calculate weighted MAE for XGBoost
xgb_weighted_mae = (
    TARGET_WEIGHTS['short'] * mean_absolute_error(y_val[:, 0], xgb_val_predictions['short']) +
    TARGET_WEIGHTS['medium'] * mean_absolute_error(y_val[:, 1], xgb_val_predictions['medium']) +
    TARGET_WEIGHTS['long'] * mean_absolute_error(y_val[:, 2], xgb_val_predictions['long'])
)

print("\n" + "="*80)
print("XGBoost Summary")
print("="*80)
print(f"Short MAE:  {mean_absolute_error(y_val[:, 0], xgb_val_predictions['short']):.6f} (weight: 0.5)")
print(f"Medium MAE: {mean_absolute_error(y_val[:, 1], xgb_val_predictions['medium']):.6f} (weight: 0.3)")
print(f"Long MAE:   {mean_absolute_error(y_val[:, 2], xgb_val_predictions['long']):.6f} (weight: 0.2)")
print(f"Weighted MAE: {xgb_weighted_mae:.6f}")
print("="*80)

Training XGBoost Models

--- Training XGBoost for target_short ---
[0]	validation_0-mae:0.00278
[100]	validation_0-mae:0.00253
[176]	validation_0-mae:0.00253
Best iteration: 126
Validation MAE: 0.002528

--- Training XGBoost for target_medium ---
[0]	validation_0-mae:0.00715
[100]	validation_0-mae:0.00633
[110]	validation_0-mae:0.00635
Best iteration: 60
Validation MAE: 0.006166

--- Training XGBoost for target_long ---
[0]	validation_0-mae:0.01572
[58]	validation_0-mae:0.01919
Best iteration: 8
Validation MAE: 0.015482

XGBoost Summary
Short MAE:  0.002528 (weight: 0.5)
Medium MAE: 0.006166 (weight: 0.3)
Long MAE:   0.015482 (weight: 0.2)
Weighted MAE: 0.006210


# Compare models
comparison = pd.DataFrame({
    'Model': ['XGBoost (3 models)', 'MLP (multi-target)'],
    'Weighted MAE': [xgb_weighted_mae, mlp_weighted_mae]
})

print(comparison.to_string(index=False))

best_model = 'XGBoost' if xgb_weighted_mae < mlp_weighted_mae else 'MLP'
print(f"\nBest model: {best_model}")

In [ ]:
print("Generating XGBoost predictions...")
test_pred_xgb_short = xgb_models['short'].predict(X_test)
test_pred_xgb_medium = xgb_models['medium'].predict(X_test)
test_pred_xgb_long = xgb_models['long'].predict(X_test)

submission_xgb = pd.DataFrame({
    'id': test_df['id'],
    'target_short': test_pred_xgb_short,
    'target_medium': test_pred_xgb_medium,
    'target_long': test_pred_xgb_long
})
submission_xgb.to_csv('submission.csv', index=False)
print(f"✅ XGBoost submission saved: submission_xgb.csv")

Generating XGBoost predictions...
✅ XGBoost submission saved: submission_xgb.csv
